# backward-on-scalar-loss composite — cx13: canonical training step: backward on scalar loss + zero_grad set_to_none

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `backward-on-scalar-loss`, `zero-grad-set-none`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
import wandb

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "backward-on-scalar-loss"
DD_ATOM_IDS = ["backward-on-scalar-loss", "zero-grad-set-none"]
DD_SUBTOPICS = ["PyTorch: backward()", "PyTorch: zero_grad"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Every PyTorch training step ends with the same four-line ritual. Two of the four lines are the atoms in this drill:

```python
loss = loss_fn(model(x), y)   # forward; scalar if loss_fn is .mean()-reducing.
loss.backward()               # atom A: backward-on-scalar-loss.
optimizer.step()              # apply update.
optimizer.zero_grad(set_to_none=True)  # atom B: zero-grad-set-none.
```

**Atom A — `backward-on-scalar-loss`.** `tensor.backward()` populates `.grad` on every leaf with `requires_grad=True` that contributed to `tensor`. The tensor MUST be a scalar (0-D / single-element) unless a `gradient=...` tensor is supplied. The canonical reduction is `.mean()` (batch-invariant scale) — `.sum()` makes the effective learning rate scale with batch size.

**Atom B — `zero-grad-set-none`.** `.backward()` ACCUMULATES into existing `.grad`. Without a reset between batches, batch N+1's gradient would be added on top of batch N's, producing a stale oversized update. `set_to_none=True` replaces `.grad` with `None` (faster, no kernel launch, no memory cleared) — the next `.backward()` allocates a fresh tensor. This is PyTorch's default since 1.7.

**Why the order matters.** `step()` reads from `.grad`. If you `zero_grad` BEFORE `step`, the optimizer sees zero gradient and the parameters never move. The order is `backward → step → zero_grad`, not `zero_grad → backward → step` (though that order works too; just don't put `zero_grad` between `backward` and `step`).

### Composite Exercise — canonical training step: backward on scalar loss + zero_grad set_to_none

**Atoms exercised together**: `backward-on-scalar-loss`, `zero-grad-set-none`

Implement `cx13_train_step(model, optimizer, x, y, loss_fn)`. ONE training step. Sequence:

1. `pred = model(x)` — forward.
2. `loss = loss_fn(pred, y)` — assume `loss_fn` already reduces to a scalar (e.g. `nn.MSELoss()` defaults to `reduction='mean'`).
3. `loss.backward()` (atom A — relies on `loss` being a scalar).
4. `optimizer.step()` — apply the update.
5. `optimizer.zero_grad(set_to_none=True)` (atom B).
6. Return `loss.item()` — the scalar Python float.

The test confirms:
- `loss.item()` is returned (NOT the tensor — wandb-loggable).
- Model parameters MOVED (so step ran after a non-None grad existed).
- After return, every parameter has `p.grad is None` (set_to_none semantics).
- Calling the function twice doesn't double-accumulate — second-call grads are NOT 2x the first-call grads (which is what would happen if `zero_grad` was missing).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx13_train_step(model, optimizer, x, y, loss_fn):
    """One canonical training step. Returns loss.item()."""
    raise NotImplementedError

def _test_cx13():
    # Case A: round-trip — loss returned as a float, params move, grads set to None.
    t.manual_seed(0)
    model = nn.Linear(3, 1)
    opt = t.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = nn.MSELoss()
    x = t.randn(8, 3)
    y = t.randn(8, 1)

    params_before = [p.detach().clone() for p in model.parameters()]
    loss_val = cx13_train_step(model, opt, x, y, loss_fn)

    assert isinstance(loss_val, float), (
        f'expected loss.item() (float); got {type(loss_val).__name__} — did you forget .item()?'
    )
    for p_before, p_after in zip(params_before, model.parameters()):
        assert not t.allclose(p_before, p_after.detach()), (
            'parameter did not move — optimizer.step() must run BEFORE zero_grad'
        )
    for p in model.parameters():
        assert p.grad is None, (
            f'p.grad must be None after step (set_to_none=True); got {p.grad!r}'
        )

    # Case B: second call works — zero_grad prevented accumulation.
    # If zero_grad was missing, the second backward would double-add gradients.
    # We verify by running TWO steps with identical (x, y) and checking that the
    # parameter delta of step 2 is close to step 1's delta (not 2x).
    t.manual_seed(1)
    m2 = nn.Linear(3, 1)
    opt2 = t.optim.SGD(m2.parameters(), lr=0.01)
    x2 = t.randn(8, 3)
    y2 = t.randn(8, 1)
    p0 = next(m2.parameters()).detach().clone()
    cx13_train_step(m2, opt2, x2, y2, loss_fn)
    p1 = next(m2.parameters()).detach().clone()
    delta1 = p1 - p0
    cx13_train_step(m2, opt2, x2, y2, loss_fn)
    p2 = next(m2.parameters()).detach().clone()
    delta2 = p2 - p1
    # delta2 should be SMALLER than 2*delta1 (because loss decreased between steps —
    # but absolutely NOT 2x as large, which signals double-accumulation).
    assert delta2.abs().max() < 2.0 * delta1.abs().max(), (
        'second-step delta ~2x first-step delta — zero_grad is not running, grads are accumulating'
    )

    # Case C: the function rejects vector losses gracefully if the loss_fn happens to be
    # misconfigured (no reduction). PyTorch raises RuntimeError on vector backward without
    # a gradient= argument.
    t.manual_seed(2)
    m3 = nn.Linear(3, 1)
    opt3 = t.optim.SGD(m3.parameters(), lr=0.1)
    vec_loss = nn.MSELoss(reduction='none')
    x3 = t.randn(8, 3)
    y3 = t.randn(8, 1)
    raised = False
    try:
        cx13_train_step(m3, opt3, x3, y3, vec_loss)
    except RuntimeError:
        raised = True
    assert raised, (
        'expected RuntimeError on backward of non-scalar loss — your code must call '
        'loss.backward() unconditionally (do not silently sum-reduce inside cx13_train_step)'
    )
    _dd_passed.add('cx13')

_test_cx13()

<details><summary>Show solution — cx13</summary>

```python
def cx13_train_step(model, optimizer, x, y, loss_fn):
    # Forward + scalar loss.
    pred = model(x)
    loss = loss_fn(pred, y)
    # Atom A: backward-on-scalar-loss. Requires loss to be a 0-D tensor.
    loss.backward()
    optimizer.step()
    # Atom B: zero-grad-set-none. set_to_none=True is the default but write it out.
    optimizer.zero_grad(set_to_none=True)
    return loss.item()
```

Returning `loss.item()` (a Python float) — not the tensor — is the wandb-friendly form: tensors aren't JSON-serializable. If your test fails on Case C, you may be accidentally calling `loss.sum().backward()` or `loss.mean().backward()` inside the function. Don't — the contract is that `loss_fn` produces a scalar; if it doesn't, propagate the error.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx13'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx13',
        'subtopics': ["PyTorch: backward()", "PyTorch: zero_grad"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()